<a href="https://colab.research.google.com/github/yaesur/business_python/blob/%EC%B2%AD%EB%85%84%EB%A7%A4%EC%9E%85%EC%9E%84%EB%8C%80%EC%A3%BC%ED%83%9D/1%EC%88%9C%EC%9C%84_%EA%B2%BD%EC%9F%81%EC%9E%90_%EC%9C%A0%EC%9E%85%EA%B3%BC_%EC%A0%90%EC%88%98_%EC%83%81%EA%B4%80%EA%B4%80%EA%B3%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  연도  1순위데이터개수   수급자수	평균점수
2021        60 		   309962	5.633333
2022       121 		   300266	5.322314
2023       152 		   299179	5.513158
2024       188 		   294595	6.101064

1순위 데이터 개수와 평균 점수 간의 피어슨 상관계수: 0.5056
**1순위 데이터 개수와 수급자 수 간의 피어슨 상관계수: -0.9819**
통계치(1순위 유입)와 1순위 점수 간의 피어슨 상관계수: -0.0762

In [6]:
import pandas as pd
import re

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

result = []

# 1. 시트별 데이터 로드 및 전처리
for i, name in enumerate(sheets):
    year_match = re.search(r'\d{4}', name)
    if not year_match:
        continue
    year = int(year_match.group())

    # 2021년부터 2024년 데이터만 수집
    if year < 2021 or year > 2024:
        continue

    # skiprows=2로 가져오되 컬럼명 검색을 위해 임시 로드
    df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)

    # [핵심 변경] 시트 내에서 '순위'와 '점수' 텍스트가 포함된 열을 자동으로 찾음
    # 만약 글자가 없다면 이전에 쓰던 인덱스 fallback 적용
    rank_col, score_col = None, None

    # 데이터의 상위 행들을 보며 '순위', '점수' 컬럼 위치 추적
    for col in df.columns:
        col_str = df[col].astype(str).str.cat(sep=' ')
        if '순위' in col_str and rank_col is None:
            rank_col = col
        elif '점' in col_str and score_col is None:
            score_col = col

    # 만약 자동으로 못 찾았다면 기존에 지정했던 인덱스 기본값 사용
    if rank_col is None or score_col is None or rank_col == score_col:
        default_levels = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
        # 점수는 보통 순위 바로 다음 열에 있으므로 1을 더해줌
        rank_col = default_levels[i] if i < len(default_levels) else 6
        score_col = rank_col + 1

    # 딱 필요한 순위와 점수 컬럼만 추출 (자치구 완전 배제)
    data = df.iloc[:, [rank_col, score_col]].copy()
    data.columns = ['순위', '점수']

    # 연도 정보 추가
    data['연도'] = year
    result.append(data)

# 모든 시트 데이터 통합
final = pd.concat(result)

# 2. 데이터 정제 (숫자만 남기기)
# 문자로 되어있는 '1순위', '6점' 등에서 숫자만 추출하는 가장 안전한 방식
final['순위'] = final['순위'].astype(str).str.extract(r'(\d+)')
final['점수'] = final['점수'].astype(str).str.extract(r'(\d+)')

# 수치형 변환
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')
final['점수'] = pd.to_numeric(final['점수'], errors='coerce')
final['연도'] = pd.to_numeric(final['연도'], errors='coerce')

# 기본 정제 후 결측치 제거
final = final.dropna(subset=['순위', '점수'])

# ⭐ 3. 1순위 데이터 필터링 및 level_num 매핑
first_priority = final[final['순위'] == 1].copy()

# 연도별 level_num 정의
level_num = {2021: 309962, 2022: 300266, 2023: 299179, 2024: 294595}
first_priority['1순위 유입'] = first_priority['연도'].map(level_num)

# 최종 분석 대상 데이터 확정
analysis_df = first_priority.dropna(subset=['1순위 유입', '점수'])

# --- 4. 분석 결과 출력 ---
print("======= 분석 결과 =======")
print(f"1순위 커트라인 분석 데이터수: {len(analysis_df)}건\n")

if len(analysis_df) > 0:
    # 1) level_num별 1순위 점수 평균
    print("[level_num 통계치별 1순위 점수 평균]")
    avg_by_level = analysis_df.groupby('1순위 유입')['점수'].mean().reset_index()
    print(avg_by_level.to_string(index=False))
    print("-" * 30)

    # 2) 상관관계 계산
    correlation = analysis_df['1순위 유입'].corr(analysis_df['점수'])
    print(f"통계치(1순위 유입)와 1순위 점수 간의 피어슨 상관계수: {correlation:.4f}")
else:
    print("⚠️ 여전히 데이터가 0건입니다. 엑셀 파일 시트의 '순위' 열에 숫자 1이 제대로 존재치 않거나 skiprows=2 범위 밖일 수 있습니다.")

======= 분석 결과 =======
1순위 커트라인 분석 데이터수: 521건

[level_num 통계치별 1순위 점수 평균]
 1순위 유입       점수
 294595 6.101064
 299179 5.513158
 300266 5.322314
 309962 5.633333
------------------------------
통계치(1순위 유입)와 1순위 점수 간의 피어슨 상관계수: -0.0762


In [12]:
import pandas as pd
import re

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

# 기존에 설정한 시트별 컬럼 인덱스 매핑 규칙 적용
level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 1. 시트별 데이터 로드 및 1순위 데이터 추출
for i, name in enumerate(sheets):
    # 시트 이름에서 연도 4자리 추출
    year_match = re.search(r'\d{4}', name)
    if not year_match:
        continue
    year = int(year_match.group())

    # 2021년부터 2024년 데이터만 수집
    if year < 2021 or year > 2024:
        continue

    df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)

    # 안전하게 행 인덱스가 있는지 확인하고 추출
    if df.shape[1] > max(level_cols[i], score_cols[i]):
        data = df.iloc[:, [level_cols[i], score_cols[i]]].copy()
        data.columns = ['순위', '점수']
        data['연도'] = year
        result.append(data)

# 데이터 통합 및 정제
final = pd.concat(result).dropna(subset=['순위'])

# '순위' 컬럼에서 '순위' 텍스트 제거 후 수치형 변환 (1순위만 필터링하기 위함)
final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')

# ✨ [핵심 변환] 오직 '1순위' 데이터만 필터링
first_priority = final[final['순위'] == 1].copy()

# --- 2. 연도별 1순위 데이터 '개수(Row Count)' 집계 ---
summary_df = first_priority.groupby('연도').size().reset_index(name='1순위데이터개수')

# --- 3. 연도별 수급자 수(level_num) 매핑 ---
level_num = {2021: 309962, 2022: 300266, 2023: 299179, 2024: 294595}
summary_df['수급자수'] = summary_df['연도'].map(level_num)

# --- 4. 최종 집계 테이블 및 상관관계 출력 ---
print("======= 연도별 집계 데이터 =======")
print(summary_df.to_string(index=False))
print("-" * 40)

# 피어슨 상관계수 계산 (데이터 개수 vs 수급자 수)
correlation = summary_df['1순위데이터개수'].corr(summary_df['수급자수'])
print(f"▶ 1순위 데이터 개수와 수급자 수 간의 피어슨 상관계수: {correlation:.4f}")

======= 연도별 집계 데이터 =======
  연도  1순위데이터개수   수급자수
2021        60 309962
2022       121 300266
2023       152 299179
2024       188 294595
----------------------------------------
▶ 1순위 데이터 개수와 수급자 수 간의 피어슨 상관계수: -0.9819


In [19]:
import pandas as pd
import re

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 1. 시트별 데이터 로드 및 추출
for i, name in enumerate(sheets):
    year_match = re.search(r'\d{4}', name)
    if not year_match:
        continue
    year = int(year_match.group())

    if year < 2021 or year > 2024:
        continue

    df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)

    if df.shape[1] > max(level_cols[i], score_cols[i]):
        data = df.iloc[:, [level_cols[i], score_cols[i]]].copy()
        data.columns = ['순위', '점수']
        data['연도'] = year
        result.append(data)

if not result:
    print("⚠️ 조건에 맞는 데이터를 찾지 못했습니다.")
else:
    # 데이터 통합
    final = pd.concat(result).dropna(subset=['순위'])

    # 전처리: '순위' 및 '점수' 컬럼 숫자형 변환
    final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
    final['순위'] = pd.to_numeric(final['순위'], errors='coerce')

    final['점수'] = final['점수'].astype(str).str.replace('점', '', regex=False).str.strip()
    final['점수'] = pd.to_numeric(final['점수'], errors='coerce')

    # 오직 '1순위' 데이터만 필터링
    first_priority = final[final['순위'] == 1].copy()

    if first_priority.empty:
        print("⚠️ '순위'가 1인 데이터가 존재하지 않습니다.")
    else:
        # --- 2. 연도별 '1순위 데이터 개수'와 '커트라인 점수 평균' 동시 집계 ---
        summary_df = first_priority.groupby('연도').agg(
            데이터개수=('순위', 'size'),
            평균점수=('점수', 'mean')
        ).reset_index()

        # --- 3. 최종 집계 테이블 출력 ---
        print("======= 연도별 데이터 개수 vs 점수 평균 =======")
        print(summary_df.to_string(index=False))
        print("-" * 50)

        # 피어슨 상관계수 계산 (데이터 개수 vs 점수 평균)
        if len(summary_df) > 1:
            correlation = summary_df['데이터개수'].corr(summary_df['평균점수'])
            print(f"▶ [결과] 1순위 데이터 개수와 평균 점수 간의 피어슨 상관계수: {correlation:.4f}")
        else:
            print("⚠️ 데이터가 부족하여 상관계수를 계산할 수 없습니다.")

======= 연도별 데이터 개수 vs 점수 평균 =======
  연도  데이터개수     평균점수
2021     60 5.633333
2022    121 5.322314
2023    152 5.513158
2024    188 6.101064
--------------------------------------------------
▶ [결과] 1순위 데이터 개수와 평균 점수 간의 피어슨 상관계수: 0.5056
